# Complete Attack Pipeline
This notebook combines:
1. **Acquisition** – camera setup, ArUco calibration, photometric calibration (from `complete_attack_v2`)
2. **Training** – adversarial patch optimisation via latent-space attack (from `single_debug_v2_untargeted`)
3. **Inference** – project the learned patch and evaluate live (from `complete_attack_v2`)

> **Configuration** is loaded from `config.yaml`. Secrets (Comet ML key) live in `.env`.
> See `README.md` for full setup instructions.

## 0 – Setup & Configuration

In [ ]:
# Load configuration and set up environment
from attack_config import load_config, setup_environment, get_comet_config

cfg = load_config("config.yaml")
on_remote = setup_environment(cfg)
print(f"on_remote = {on_remote}")

if not on_remote:
    import os
    os.makedirs('./captures_frames_multiview', exist_ok=True)

# Set to True to skip Section 1 (Acquisition) and reuse saved calibration state
SKIP_ACQUISITION = False

In [ ]:
# Optional: Comet ML experiment tracking
from experiment_tracking import ExperimentTracker

comet_cfg = get_comet_config(cfg)
tracker = ExperimentTracker(comet_cfg).start()

In [ ]:
%matplotlib inline

## 1 – Acquisition (Camera + Calibration)
Set up the capture system, detect ArUco markers, and run photometric calibration.

**Controls during `display_drawer()`:**
- Draw the projection rectangle on the fullscreen window, then:
  - Press **`c`** – starts capturing training frames (multi-view). Move the object / camera around to collect diverse viewpoints.
  - Press **`ESC`** – skips capture and continues to calibration.
- During frame capture, press **`q`** to stop recording.

In [ ]:
from capture_utils_v2 import CaptureSystem

system = CaptureSystem()

In [ ]:
if not SKIP_ACQUISITION:
    # Display the projector drawer for visual alignment
    system.display_drawer()

In [ ]:
if not SKIP_ACQUISITION:
    # Run ArUco corner detection
    system.run_aruco_detector()
    # system.run_aruco_detector_manual()  # alternative manual mode

In [ ]:
if not SKIP_ACQUISITION:
    # Run photometric calibration
    ret = system.photometric_calibration(return_captured=True)

In [ ]:
import pickle

if not SKIP_ACQUISITION:
    # Persist the calibration state so training can resume later
    with open("capture_system_state.pkl", "wb") as f:
        pickle.dump({
            'orig_proj_corners': system.orig_proj_corners,
            'corners_img_proj': system.corners_img_proj,
            'orig_img': system.orig_img,
            'img_non_zero_section': system.img_non_zero_section,
            'H': system.H,
        }, f)
    print("Capture state saved.")
else:
    # Load previously saved calibration state
    with open("capture_system_state.pkl", "rb") as f:
        state = pickle.load(f)

    class _Namespace:
        pass

    system = CaptureSystem()
    system.orig_proj_corners = state['orig_proj_corners']
    system.corners_img_proj = state['corners_img_proj']
    system.orig_img = state['orig_img']
    system.img_non_zero_section = state['img_non_zero_section']
    system.H = state['H']
    print("Capture state loaded from capture_system_state.pkl.")

## 2 – Training
Load frames, build dataset, and run the adversarial latent-space optimisation.

In [ ]:
# Load classifiers according to config
from classifier_loader import setup_classifiers

clfs = setup_classifiers(cfg)
predict_raw      = clfs["predict_raw"]
predict_raw_dev  = clfs["predict_raw_dev"]
predict_raw_test = clfs["predict_raw_test"]
weights          = clfs["weights"]
orig_clases      = clfs["orig_clases"]
model_name       = clfs["model_name"]

print(f"Train  classifier : {clfs['model_name']}")
print(f"Dev    classifier : {clfs['model_dev'].__class__.__name__}")
print(f"Test   classifier : {clfs['model_test'].__class__.__name__}")
print(f"Forbidden classes : {orig_clases}")

In [ ]:
# Load photometric calibration data, frames, and homographies
from data_preparation import load_photometric_calibration, load_frames_and_homographies, build_dataloaders

calib_dir = cfg.get("capture", {}).get("photometric_calibrations_dir", "./photometric_calibrations")
calib_data, height, width = load_photometric_calibration(calib_dir)

valid_frames, Hs = load_frames_and_homographies(
    cfg, predict_raw, weights, orig_clases,
    height, width, on_remote=on_remote,
)

In [ ]:
# Build data loaders
tcfg = cfg.get("training", {})
train_loader, val_loader, test_loader = build_dataloaders(
    valid_frames, Hs,
    train_split=tcfg.get("train_split", 0.8),
    val_split=tcfg.get("val_split", 0.1),
)
print(f"Train batches: {len(train_loader)} | Val: {len(val_loader.dataset)} | Test: {len(test_loader.dataset)}")

In [ ]:
# Load VAE
from vae_utils import load_vae
device = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
vae = load_vae(device)
print("VAE loaded.")

In [ ]:
# Prepare the photometric augmentor model
augmentor_model = calib_data['augmentor'].to(device).eval()

In [ ]:
# Re-attach Comet experiment (ensures code capture)
tracker.reattach()

In [ ]:
# Run the training loop
from training import train_adversarial_patches

train_results = train_adversarial_patches(
    cfg=cfg,
    train_loader=train_loader,
    val_loader=val_loader,
    valid_frames=valid_frames,
    height=height,
    width=width,
    orig_clases=orig_clases,
    predict_raw=predict_raw,
    predict_raw_dev=predict_raw_dev,
    weights=weights,
    augmentor_model=augmentor_model,
    tracker=tracker,
    model_name=model_name,
    device=device,
)

print(f"\nBest success rate : {train_results['best_success_rate']:.1%}")
print(f"Best loss         : {train_results['best_loss']:.3f}")

In [ ]:
# Save ablation CSVs
from evaluation import save_ablation_csvs, save_top_patches

save_ablation_csvs(train_results, cfg, tracker)
save_dir, all_patches, performance = save_top_patches(train_results, height, width, device)

In [ ]:
# Visualise best patches
from visualization import plot_best_patch_summary

plot_best_patch_summary(all_patches, performance)

## 3 – Inference
Project the best patch onto the scene and classify the live camera feed.

In [ ]:
import cv2
import numpy as np
import os

# Pick the best patch image from the saved directory
best_idx, best_rate = performance[0]
# Find the matching file
import glob as _g
best_files = _g.glob(os.path.join(save_dir, f"{best_idx}_*.png"))
if best_files:
    im_path = best_files[0]
else:
    # Fallback: pick the first PNG in save_dir
    im_path = _g.glob(os.path.join(save_dir, "*.png"))[0]

image_to_project = cv2.cvtColor(cv2.imread(im_path), cv2.COLOR_BGR2RGB)
print(f"Projecting patch: {im_path}")

In [ ]:
# Display the patch on the projector
system.plot_on_screen(image_to_project)

In [ ]:
import torch
from PIL import Image, ImageDraw, ImageFont

def add_text_with_background(img, text, position, font_size=24,
                              text_color=(255, 255, 255), bg_color=(0, 0, 0)):
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(img_rgb)
    draw = ImageDraw.Draw(pil_img)
    try:
        font = ImageFont.truetype("arial.ttf", font_size)
    except Exception:
        try:
            font = ImageFont.truetype("C:/Windows/Fonts/arial.ttf", font_size)
        except Exception:
            font = ImageFont.load_default()
    bbox = draw.textbbox(position, text, font=font)
    pad = 5
    draw.rectangle([bbox[0]-pad, bbox[1]-pad, bbox[2]+pad, bbox[3]+pad], fill=bg_color)
    draw.text(position, text, font=font, fill=text_color)
    return cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)

tt_infer = lambda x: torch.tensor(cv2.cvtColor(x, cv2.COLOR_BGR2RGB)/255.).permute(2,0,1).float()

infer_cfg = cfg.get("inference", {})
num_infer_frames = infer_cfg.get("num_frames", 2400)
cap = system.cap

caps, caps_with_text, results, raw_predictions = [], [], [], []
for i in range(num_infer_frames):
    r = cap.read()[1]
    tr = tt_infer(r)
    with torch.no_grad():
        p = predict_raw(tr.unsqueeze(0).cuda())
        res = weights.meta["categories"][p[0].argmax(0).item()]
        prob = p[0].max(0).values.item() * 100
        raw_predictions.append(p)
    r_orig = r.copy()
    r = add_text_with_background(r, f'Pred: {res}: {prob:.2f}%', (10, 10), font_size=28)
    cv2.imshow('frame', r)
    results.append(res)
    caps_with_text.append(r)
    caps.append(r_orig)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cv2.destroyAllWindows()
print(f"Captured {len(caps)} inference frames.")

In [ ]:
# Save inference captures
import datetime as _dt

infer_caps_dir = infer_cfg.get("infer_caps_dir", os.path.join(cfg.get("results_dir", "./results"), "infer_caps"))
cur_time = _dt.datetime.now().strftime("%Y-%m-%d_%H_%M")
exp_name = os.path.basename(im_path).split('.')[0]
out_dir = os.path.join(infer_caps_dir, exp_name, cur_time)
os.makedirs(out_dir, exist_ok=True)

for name, obj in [("caps", caps), ("caps_with_text", caps_with_text),
                   ("results", results), ("raw_predictions", raw_predictions)]:
    with open(os.path.join(out_dir, f"{name}.pkl"), "wb") as f:
        pickle.dump(obj, f)
print(f"Inference results saved → {out_dir}")

## 3.1 – Tracked Inference
Project the best patch while **tracking** the printed ArUco marker so the projection follows board movement in real time.  
Classification results are collected the same way as in §3.

In [ ]:
from tracking_utils import TrackerSystem

# Printed ArUco marker ID on the physical board
printed_aruco_id = cfg.get("tracking", {}).get("printed_aruco_id", 10)

tracker_sys = TrackerSystem(
    system,
    predict_raw,
    weights,
    printed_aruco_id=printed_aruco_id,
)

In [ ]:
# Run tracked projection + live classification
# Press 'q' to stop, 'r' to recalibrate
tracked_caps, tracked_results = tracker_sys.track_project_and_classify(image_to_project)
print(f"Tracked inference: {len(tracked_caps)} frames captured.")

In [ ]:
# Save tracked inference results
import datetime as _dt

tracked_dir = os.path.join(
    cfg.get("results_dir", "./results"),
    "tracked_infer_caps",
    os.path.basename(im_path).split('.')[0],
    _dt.datetime.now().strftime("%Y-%m-%d_%H_%M"),
)
os.makedirs(tracked_dir, exist_ok=True)

for name, obj in [("caps", tracked_caps), ("results", tracked_results)]:
    with open(os.path.join(tracked_dir, f"{name}.pkl"), "wb") as f:
        pickle.dump(obj, f)
print(f"Tracked inference results saved → {tracked_dir}")

## 4 – Cleanup

In [ ]:
tracker.end()
cv2.destroyAllWindows()
print("Done.")